In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd 
import config 
import era5_clean as ec

In [2]:
ds=ec.load_raw_era5()
ds

Opened 1 of 41 file(s)
Dimensions: {'valid_time': 744, 'latitude': 31, 'longitude': 21}
Variables : ['t2m', 'tp', 'swvl1', 'u10', 'v10']


<xarray.Dataset> Size: 10MB
Dimensions:     (valid_time: 744, latitude: 31, longitude: 21)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 6kB 2020-01-01 ... 2020-01-31T23:...
    expver      (valid_time) <U4 12kB ...
  * latitude    (latitude) float64 248B 20.0 19.9 19.8 19.7 ... 17.2 17.1 17.0
  * longitude   (longitude) float64 168B 98.0 98.1 98.2 98.3 ... 99.8 99.9 100.0
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 2MB ...
    tp          (valid_time, latitude, longitude) float32 2MB ...
    swvl1       (valid_time, latitude, longitude) float32 2MB ...
    u10         (valid_time, latitude, longitude) float32 2MB ...
    v10         (valid_time, latitude, longitude) float32 2MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-27T05:15 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:
hourly=ec.to_dataframe(ds)
ds

<xarray.Dataset> Size: 10MB
Dimensions:     (valid_time: 744, latitude: 31, longitude: 21)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 6kB 2020-01-01 ... 2020-01-31T23:...
    expver      (valid_time) <U4 12kB '0001' '0001' '0001' ... '0001' '0001'
  * latitude    (latitude) float64 248B 20.0 19.9 19.8 19.7 ... 17.2 17.1 17.0
  * longitude   (longitude) float64 168B 98.0 98.1 98.2 98.3 ... 99.8 99.9 100.0
    number      int64 8B 0
Data variables:
    t2m         (valid_time, latitude, longitude) float32 2MB 288.3 ... 293.5
    tp          (valid_time, latitude, longitude) float32 2MB 0.001242 ... 4....
    swvl1       (valid_time, latitude, longitude) float32 2MB 0.3347 ... 0.1211
    u10         (valid_time, latitude, longitude) float32 2MB -0.5741 ... -0....
    v10         (valid_time, latitude, longitude) float32 2MB -0.256 ... 0.8785
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-27T05:15 GRIB to CDM+CF via cfgrib-0.9.1...

In [4]:
print(f"300 K      -> {ec.kelvin_to_celsius(300):.2f} °C   (expected: 26.85)")
print(f"0.024 m    -> {ec.metres_to_mm(0.024):.1f} mm   (expected: 24.0)")
print(f"10 m/s     -> {ec.ms_to_kmh(10):.1f} km/h (expected: 36.0)")

300 K      -> 26.85 °C   (expected: 26.85)
0.024 m    -> 24.0 mm   (expected: 24.0)
10 m/s     -> 36.0 km/h (expected: 36.0)


In [5]:
wind_cmp=ec.compare_wind_methods(hourly)
wind_cmp.head(10)

,grid_id,date,speed_correct_ms,speed_wrong_ms,difference_ms,understated_by_pct
0,17.0_100.0,2020-01-01,1.468236,1.110301,0.358,24.400000
1,17.0_100.0,2020-01-02,1.088444,0.682900,0.406,37.299999
2,17.0_100.0,2020-01-03,0.921346,0.366751,0.555,60.200001
3,17.0_100.0,2020-01-04,0.828906,0.474363,0.355,42.799999
4,17.0_100.0,2020-01-05,1.013950,0.794765,0.219,21.600000
5,17.0_100.0,2020-01-06,1.149937,0.966733,0.183,15.900000
6,17.0_100.0,2020-01-07,1.140420,0.726197,0.414,36.299999
7,17.0_100.0,2020-01-08,0.759837,0.167110,0.593,78.000000
8,17.0_100.0,2020-01-09,0.830553,0.545075,0.285,34.400002
9,17.0_100.0,2020-01-10,0.755409,0.507477,0.248,32.799999


In [6]:
print(f"Average difference: {wind_cmp['understated_by_pct'].mean():.1f}%")
print(f"Maximum difference: {wind_cmp['understated_by_pct'].max():.1f}%")

worst = wind_cmp.loc[wind_cmp["understated_by_pct"].idxmax()]

print()
print("Day with the largest difference:")
print(f"  Correct: {worst.speed_correct_ms:.2f} m/s")
print(f"  Incorrect: {worst.speed_wrong_ms:.2f} m/s")

Average difference: 37.0%
Maximum difference: 99.4%

Day with the largest difference:
  Correct: 0.73 m/s
  Incorrect: 0.00 m/s


In [7]:
daily=ec.aggregate_daily(hourly)
daily.head()

Hourly rows 484,344 -> daily rows 20,181
Columns: ['grid_id', 'date', 'soil_moisture', 'latitude', 'longitude', 'temp_max_c', 'temp_mean_c', 'rainfall_mm', 'wind_speed_kmh', 'year', 'month']


,grid_id,date,soil_moisture,latitude,longitude,temp_max_c,temp_mean_c,rainfall_mm,wind_speed_kmh,year,month
0,17.0_100.0,2020-01-01,0.123147,17.0,100.0,31.516022,26.694763,0.003666,5.285648,2020,1
1,17.0_100.0,2020-01-02,0.122920,17.0,100.0,31.729645,26.266785,0.003660,3.918399,2020,1
2,17.0_100.0,2020-01-03,0.122693,17.0,100.0,32.332184,26.812012,0.001720,3.316846,2020,1
3,17.0_100.0,2020-01-04,0.122496,17.0,100.0,32.310944,27.081970,0.002578,2.984062,2020,1
4,17.0_100.0,2020-01-05,0.122351,17.0,100.0,32.081940,26.599182,0.002581,3.650221,2020,1


In [8]:
rain_cmp=ec.compare_rainfall_methods(hourly)
rain_cmp[rain_cmp['max_within_day_mm'] > 0].head(10)

,grid_id,date,sum_all_hours_mm,max_within_day_mm,value_at_23h_mm,sum_overstates_by_x
0,17.0_100.0,2020-01-01,0.033978,0.003666,0.003666,9.300000
1,17.0_100.0,2020-01-02,0.018239,0.003660,0.000882,20.700001
2,17.0_100.0,2020-01-03,0.029151,0.001720,0.001705,17.100000
3,17.0_100.0,2020-01-04,0.019745,0.002578,0.002578,7.700000
4,17.0_100.0,2020-01-05,0.015403,0.002581,0.001719,9.000000
5,17.0_100.0,2020-01-06,0.013682,0.001719,0.000852,16.100000
6,17.0_100.0,2020-01-07,0.000852,0.000852,0.000000,NaN
8,17.0_100.0,2020-01-09,0.005189,0.000441,0.000441,11.800000
9,17.0_100.0,2020-01-10,0.000441,0.000441,0.000000,NaN
12,17.0_100.0,2020-01-13,0.042993,0.005788,0.005788,7.400000


In [9]:
daily=ec.add_rolling_features(daily)
one_cell = daily[daily.grid_id == daily.grid_id.iloc[0]]
one_cell[["date","rainfall_mm","rainfall_7d_mm"]].head(10)

Added rainfall_7d_mm
  3,906 row(s) have less than 7 days of history


,date,rainfall_mm,rainfall_7d_mm
0,2020-01-01,0.003666,0.003666
1,2020-01-02,0.003660,0.007325
2,2020-01-03,0.001720,0.009045
3,2020-01-04,0.002578,0.011623
4,2020-01-05,0.002581,0.014205
5,2020-01-06,0.001719,0.015924
6,2020-01-07,0.000852,0.016776
7,2020-01-08,0.000000,0.013111
8,2020-01-09,0.000441,0.009892
9,2020-01-10,0.000441,0.008613


In [10]:
wrong=daily.sort_values("date").copy()
wrong["rainfall_7d_WRONG"] = wrong["rainfall_mm"].rolling(7, min_periods=1).sum()

check = wrong.sort_values(["grid_id","date"]).groupby("grid_id").head(3)
check[["grid_id","date","rainfall_mm","rainfall_7d_mm","rainfall_7d_WRONG"]]

,grid_id,date,rainfall_mm,rainfall_7d_mm,rainfall_7d_WRONG
0,17.0_100.0,2020-01-01,0.003666,0.003666,0.003666
1,17.0_100.0,2020-01-02,0.003660,0.007325,0.434938
2,17.0_100.0,2020-01-03,0.001720,0.009045,0.119784
31,17.0_98.0,2020-01-01,0.002146,0.002146,1.210350
32,17.0_98.0,2020-01-02,0.002140,0.004286,0.695568
...,...,...,...,...,...
20120,20.0_99.8,2020-01-02,0.008482,0.147527,0.455555
20121,20.0_99.8,2020-01-03,0.031179,0.178707,0.187922
20150,20.0_99.9,2020-01-01,0.045913,0.045913,1.804742
20151,20.0_99.9,2020-01-02,0.009465,0.055379,0.631985


In [11]:
daily=ec.add_dry_days(daily)
one_cell = daily[daily.grid_id == daily.grid_id.iloc[0]]
one_cell[["date","rainfall_mm","dry_days"]].head(12)

Added dry_days (threshold 1.0 mm)
  longest dry spell: 31 days


,date,rainfall_mm,dry_days
0,2020-01-01,0.003666,1
1,2020-01-02,0.003660,2
2,2020-01-03,0.001720,3
3,2020-01-04,0.002578,4
4,2020-01-05,0.002581,5
5,2020-01-06,0.001719,6
6,2020-01-07,0.000852,7
7,2020-01-08,0.000000,8
8,2020-01-09,0.000441,9
9,2020-01-10,0.000441,10


In [12]:
daily_full,report=ec.clean_era5_folder()
daily_full.head()


  [1/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 20,181 daily rows
  [2/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 18,879 daily rows
  [3/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 20,181 daily rows
  [4/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 19,530 daily rows
  [5/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 20,181 daily rows
  [6/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 19,530 daily rows
  [7/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 20,181 daily rows
  [8/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 20,181 daily rows
  [9/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 19,530 daily rows
  [10/41] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_... -> 20,181 daily rows
  [11/41] era5land_10m_u_component_of_wind_10m_v_component_

,grid_id,date,soil_moisture,latitude,longitude,temp_max_c,temp_mean_c,rainfall_mm,wind_speed_kmh,year,month,rainfall_7d_mm,dry_days
0,17.0_100.0,2020-01-01,0.123147,17.0,100.0,31.516022,26.694763,0.003666,5.285648,2020,1,0.003666,1
1,17.0_100.0,2020-01-02,0.122920,17.0,100.0,31.729645,26.266785,0.003660,3.918399,2020,1,0.007325,2
2,17.0_100.0,2020-01-03,0.122693,17.0,100.0,32.332184,26.812012,0.001720,3.316846,2020,1,0.009045,3
3,17.0_100.0,2020-01-04,0.122496,17.0,100.0,32.310944,27.081970,0.002578,2.984062,2020,1,0.011623,4
4,17.0_100.0,2020-01-05,0.122351,17.0,100.0,32.081940,26.599182,0.002581,3.650221,2020,1,0.014205,5


In [13]:
import schema

present = [c for c in daily_full.columns if c in schema.SCHEMA]
missing = [c for c in schema.SCHEMA if c not in daily_full.columns]

print(f"Columns matching the schema: {len(present)}")
print(f"Still missing              : {missing}")
print()
print("(The missing columns will be added in Sessions 09–11)")

Columns matching the schema: 13
Still missing              : ['country', 'fire', 'hotspot_count', 'frp_max', 'frp_sum', 'land_cover_dominant', 'forest_pct', 'cropland_pct', 'grassland_pct', 'other_pct']

(The missing columns will be added in Sessions 09–11)


In [14]:
path=ec.save_daily(daily_full)
reloaded=pd.read_parquet(path)
print(f"Reloaded: {len(reloaded):,} rows")

Saved 811,797 rows -> D:\Duanduhoche.Huy\wildfire-risk-analysis\data\processed\weather_daily.parquet
Reloaded: 811,797 rows
